# NVIDIA Riva — Full speech suite (client)

Riva runs on a separate Triton-backed server (deployed by TELUS or self-hosted). This notebook is a client that talks to the server URL.

Reference: https://docs.nvidia.com/deeplearning/riva/user-guide/docs/quick-start-guide/local.html

Set `RIVA_URI` env var or replace below to point at the deployed Riva server.

In [ ]:
import os, riva.client

RIVA_URI = os.environ.get('RIVA_URI', 'localhost:50051')
auth = riva.client.Auth(uri=RIVA_URI)
asr_service = riva.client.ASRService(auth)
print(f'Riva ASR service connected to {RIVA_URI}')

In [ ]:
config = riva.client.RecognitionConfig(
    encoding=riva.client.AudioEncoding.LINEAR_PCM,
    language_code='en-US',
    max_alternatives=1,
    enable_automatic_punctuation=True,
    audio_channel_count=1,
)

with open('/workspace/data/sample.wav', 'rb') as f:
    audio_bytes = f.read()
response = asr_service.offline_recognize(audio_bytes, config)
for r in response.results:
    print(r.alternatives[0].transcript)

## TTS (Text → Speech) via Riva

Riva also exposes TTS — single Triton server, low latency, supports speaker IDs + voice cloning.

In [ ]:
tts_service = riva.client.SpeechSynthesisService(auth)
resp = tts_service.synthesize(
    text='Welcome to the Indigenomics AI Creator Tech Jam.',
    voice_name='English-US.Female-1',
    language_code='en-US',
    encoding=riva.client.AudioEncoding.LINEAR_PCM,
    sample_rate_hz=44100,
)
with open('/workspace/data/tts-out.wav', 'wb') as f:
    f.write(resp.audio)
print('TTS written to /workspace/data/tts-out.wav')